## Module 2-2 Data Visualization

<img src="https://matplotlib.org/stable/_images/sphx_glr_logos2_003.png" width="300"/> 

[Matplotlib Tutorial](https://matplotlib.org/stable/tutorials/index) 

<img src="https://seaborn.pydata.org/_static/logo-wide-lightbg.svg" width="300"/> 

[Seaborn Tutorial](https://seaborn.pydata.org/tutorial.html)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/comp_sample.csv")
df['mv'] = df['prcc_f'] * df['csho']
df['size'] = np.log(df['at'])
df['roa'] = df['ni'] / df['at']
df = df.loc[(df['mv'] > 0) & (df['size'] > 0), ['gvkey', 'fyear', 'state', 'at', 'ni', 'oancf', 'mv', 'size', 'roa']].dropna().copy()

In [ ]:
df['logmv'] = np.log(df['mv'])

df['Large'] = df.groupby('fyear')['logmv'].transform(lambda x: x > x.median()).astype(int)
df['Size_Quintile'] = df.groupby('fyear')['logmv'].transform(lambda x: pd.qcut(x, 10, labels=False))

In [ ]:
#Truncate the variables
for i in ['size', 'logmv', 'roa']:
    df[i] = df.groupby('fyear')[i].transform(lambda x: x.mask((df[i] > df[i].quantile(0.99)) | (df[i] < df[i].quantile(0.01))))
df.dropna(inplace=True)

### 0. Basic Set-ups

#### 0.1 Canvas

In [ ]:
plt.figure(figsize=(16, 9))
df['at'].hist(bins=100, color='blue', alpha=0.7)
plt.show()

#### 0.2. Colors

In [ ]:
sns.color_palette('Set3')

In [ ]:
green = sns.color_palette('Set3')[6]
blue = sns.color_palette('Set3')[4]
red = sns.color_palette('Set3')[3]
yellow = sns.color_palette('Set3')[1]

In [ ]:
green

In [ ]:
mpl.colors.ListedColormap([green, blue, red, yellow], name='Color Palette')

There are many color selectors available online, such as [this one provided by Google](https://share.google/pppBdvc6UUtJrjsxq). You can copy the **HEX code** or **RGB code** to define the color you want. ([details](https://en.wikipedia.org/wiki/Web_colors))

In [ ]:
sky = '#87CEEB'

<div style="width:80px;height:45px;background-color:#87CEEB;"></div>

In [ ]:
fire = (242/256, 93/256, 2/256)

<div style="width:80px; height:45px; background-color: rgb(242, 93, 2);"></div>

### 1. Visualize Distribution of a Single Variable

In [ ]:
Asset = df['size']

print(f"Mean: {Asset.mean()}")
print(f"Median: {Asset.median()}")
print(f"Variance: {Asset.var()}")
print(f"Standard Deviation: {Asset.std()}")

print(f"Mode: {Asset.mode()[0]}") #  The value that appears most frequently in the vairiable
print(f"Range: {Asset.max() - Asset.min()}")
print(f"IQR: {Asset.quantile(0.75) - Asset.quantile(0.25)}")

print(f"Skewness: {Asset.skew()}")
print(f"Kurtosis: {Asset.kurt()}")

#### 1.1. KDE Plot (Kernel Density Estimate)
A KDE plot provides a smooth estimate of the distribution.

You can read [this article](https://towardsdatascience.com/kernel-density-estimation-explained-step-by-step-7cc5b5bc4517/) for more details about how KDE is produced.

In [ ]:
plt.figure(figsize=(8, 4.5))

sns.kdeplot(data = df, 
            x = 'size', 
            fill=True, 
            color=green, 
            alpha=0.1, 
            linewidth=2)
# Equivalent to `sns.kdeplot(df['size'], fill=True, color=green, alpha=0.1, linewidth=2)`

plt.title('KDE Plot')
plt.xlabel('Log Total Assets')
plt.ylabel('Density')
plt.show()

#### 1.2. Histogram

In [ ]:
plt.figure(figsize=(8, 4.5))

sns.histplot(data = df, 
             x = 'size', 
             bins = 50, 
             kde = True, # Add a KDE curve to the histogram
             edgecolor = 'black', # Add black edges to the bars 
             color=green)

plt.title('Histogram')
plt.xlabel('Log Total Assets', fontsize=16, labelpad=10)
plt.ylabel('Frequency', fontsize=16, rotation=90, labelpad=10)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4.5))

sns.histplot(data=df, x='size', kde=True, color=green, edgecolor='black', label='size')
sns.histplot(data=df, x='logmv', kde=True, color=red, edgecolor='black', label='logmv')

plt.legend(title='Variable')
plt.xlabel("")
plt.ylabel("Frequency")
plt.title("Data Distribution with Mean and Standard Deviation")
plt.show()

In [ ]:
# Create a 2x3 grid of subplots
fig, axes = plt.subplots(1,3, figsize=(9, 3))

# Flatten the axes array for easy iteration
axes = axes.flatten()

var_list = ['size', 'logmv', 'roa']
# Plot histograms
for i, col in enumerate(var_list):
    sns.histplot(ax = axes[i],
            data = df, x = col, 
            bins=50, kde=True, edgecolor='k', color=green)
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

# Adjust layout
plt.tight_layout()
# Show the plot
plt.show()

#### 1.3. Box Plot

Also known as a box-and-whisker plot.

| Component                | Description                                                               |
| ------------------------ | ------------------------------------------------------------------------- |
| Box                  | Spans from Q1 to Q3 (25th to 75th percentile) → This is the IQR   |
| Line inside the box  | The median (Q2), i.e., 50th percentile                                |
| Whiskers             | Extend to the smallest and largest values within 1.5 × IQR from Q1/Q3 |
| Outliers             | Data points outside the whiskers — often shown as dots or stars       |


In [ ]:
plt.figure(figsize=(8, 4.5))

sns.boxplot(data = df, x = 'logmv', 
            color=blue, 
            width=0.5, 
            linewidth=2,
            fliersize=5)
# use `y = 'size'` for vertical box plot

plt.title("Box Plot of Data")
plt.xlabel('Size')
plt.show()

#### 1.4. Violin Plot
A violin plot combines KDE and box plots, showing the distribution's shape.

In [ ]:
# Violin plot
plt.figure(figsize=(16, 9))

sns.violinplot(data = df, x = 'size', hue='Large', palette= [blue, red], linewidth=2, inner='quartile')

plt.title('Violin Plot')
plt.legend(['Small', 'Large'])
plt.xlabel('Value')
plt.ylabel('Market Value (log scale)')
plt.show()

#### 1.5. ECDF (Empirical Cumulative Distribution Function) Plot
An ECDF plot shows the cumulative distribution of a variable.

In [ ]:
plt.figure(figsize=(8, 4.5))

sns.ecdfplot(data = df, x = 'size', color=blue, linewidth=2)

plt.title('ECDF Plot')
plt.xlabel('Value')
plt.ylabel('Cumulative Probability')
plt.show()

#### 1.6. Q-Q Plot
A Q-Q plot compares the variable’s distribution to a theoretical distribution, such as normal.

In [ ]:
import scipy.stats as stats

In [ ]:
plt.figure(figsize=(8, 4.5))

stats.probplot(df['size'], dist="norm", plot=plt)

plt.title('Q-Q Plot')
plt.show()

### 2. Plotting Trend of a Variable

#### 2.1. Trendline scatter plot using `plt.plot()`

In [ ]:
plt.figure(figsize=(12, 7))

plt.plot(df.groupby('fyear')['size'].mean(), "o", linestyle = "-", color=fire, label="Size")
plt.plot(df.groupby('fyear')['logmv'].mean(), "d", linestyle = "--", color=green, label="MV")

plt.legend(loc="upper right", shadow=True, fontsize="medium")
plt.xlabel("Year", fontsize=16, labelpad=10)
plt.show()

#### 2.2. Line Plot using `sns.lineplot()`

In [ ]:
plt.figure(figsize=(12, 7))

sns.lineplot(df.groupby('fyear')['size'].mean(), label="Size", color=fire)
sns.lineplot(df.groupby('fyear')['logmv'].mean(), label="MV", color=green)

plt.legend(loc="upper right", shadow=True, fontsize="medium")
plt.xlabel("Year", fontsize=16, labelpad=10)
plt.show()

### 3. Compare the Distributions of a Variable in subsamples

#### 3.1. Pie Chart and Bar Chart

In [ ]:
# Pie Chart
df_sample = (df.loc[df['state'].isin(['CA', 'TX', 'NY', 'ON', 'MA', 'BC', 'FL'])]
             .groupby('state')['mv'].sum()
             .reset_index()
            )

plt.figure(figsize=(7, 7))

plt.pie(x=df_sample["mv"], labels=df_sample["state"], autopct='%1.1f%%', colors=sns.color_palette('Set3'), startangle=140)

plt.legend()
plt.title("Market Cap by State")
plt.show()

In [ ]:
# Bar Chart
plt.figure(figsize=(8, 4.5))

sns.barplot(data=df.loc[df['state'].isin(['CA', 'TX', 'NY', 'ON', 'MA', 'BC', 'FL'])],
             x="mv", y="state", 
             orient="h", 
             color=fire, 
             edgecolor='red', 
             linewidth=1)

plt.title("Market Cap by State")
plt.show()

#### 3.1. Using Two Histplots

In [ ]:
# Generate random data for normal and skewed distributions
normal_data = np.random.normal(loc=df['size'].mean(), scale=df['size'].std(), size=len(df))
skewed_data = np.random.exponential(scale=df['size'].std(), size=len(df))

In [ ]:
# Plot distributions
plt.figure(figsize=(12, 6))

sns.histplot(data = normal_data, kde=True, color=green, label='Normal Distribution')
sns.histplot(data = df, x = 'size', kde=True, color=red, label='Real Log Asset')

plt.legend()
plt.title("Asset vs. Normal Distribution")
plt.show()

#### 3.2. Using KDE Plots

In [ ]:
plt.figure(figsize=(8, 4.5))

sns.kdeplot(data=df, x = 'roa', 
            hue='Large', 
            fill=True, 
            palette = sns.color_palette('Set2')[3:5], 
            alpha=0.5, 
            linewidth=2)

plt.legend(['Small', 'Large'])
plt.title('KDE Plot')
plt.xlabel('Value')
plt.ylabel('Density')
plt.show()

#### 3.3. Using Violin plots

In [ ]:
plt.figure(figsize=(16, 9))

sns.violinplot(data = df, x = 'size', hue='Large', palette= [fire, sky], linewidth=2, inner='quartile')

plt.title('Violin Plot')
plt.legend(['Small', 'Large'])
plt.xlabel('Value')
plt.ylabel('Market Value (log scale)')
plt.show()

### 4. Correlation between Two Variables

#### 4.1. Scatter Plot

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(data = df.sample(1000), x='size', y='logmv', color = blue, alpha=0.5)

plt.title(f"Scatter Plot", fontsize=16)
plt.xlabel('Size', fontsize=12, labelpad=10)
plt.ylabel('Log Market Value', fontsize=12, labelpad=10)
plt.show()

#### 4.2. Regression Plot

In [ ]:
correlation = np.corrcoef(df['size'], df['logmv'])[0, 1]
print(f"Pearson Correlation: {correlation}")

In [ ]:
# Scatter plot with regression line
plt.figure(figsize=(8, 6))

sns.regplot(data = df.sample(1000), x='size', y='logmv', color = blue, line_kws={'color': 'black', 'alpha': 0.5}, ci = None, scatter_kws={'alpha': 0.1})

plt.title(f"Scatter Plot with Pearson Correlation: {np.corrcoef(df['size'], df['logmv'])[0, 1]:.3f}", fontsize=16)
plt.xlabel('Size', fontsize=12, labelpad=10)
plt.ylabel('Log Market Value', fontsize=12, labelpad=10)
plt.show()

#### 4.3. Bivariate Distribution using 2D KDE Plot

In [ ]:
plt.figure(figsize=(8, 4.5))

sns.kdeplot(
    data=df.sample(1000),
    x='size',
    y='logmv',
    cmap='Blues',
    fill=True
)

plt.xlim(1, 12)
plt.ylim(1, 12)
plt.title('2D KDE Plot')
plt.xlabel('Total Assets (log scale)')
plt.ylabel('MV (log scale)')
plt.show()